# บทที่ 9 การประเมินแบบจำลองและการระบุความไม่แน่นอน
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch09_evaluation/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch09_evaluation/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch09_evaluation/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch09_evaluation'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'arviz': 'arviz>=0.17', 'matplotlib': 'matplotlib', 'numpy': 'numpy', 'pymc': 'pymc>=5.10', 'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.3'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_blocked_cv.py` — โค้ดที่ 9.1

Ch.9  Shuffled vs blocked cross-validation on time-correlated data, and a new independent run.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_blocked_cv.py  จากโฟลเดอร์ของบท
__file__ = '01_blocked_cv.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Shuffled vs blocked cross-validation on time-correlated data, and a new independent run.

Book references: ssec:kfold-cv, code:blocked-cv (line for line)
SYNTHETIC AR(1) data (seed 2026; new run seed 7).
"""
import os
from pathlib import Path

os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 9.1

In [ ]:
# --- code:blocked-cv ---------------------------------------------------------
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

def ar1(n, phi, sigma, rng):
    """AR(1) process: e[i] = phi * e[i-1] + white noise."""
    e = np.empty(n)
    e[0] = rng.normal(scale=sigma / np.sqrt(1 - phi**2))
    for i in range(1, n):
        e[i] = phi * e[i - 1] + rng.normal(scale=sigma)
    return e

def make_run(n=2000, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n) * 1.0
    x = ar1(n, 0.995, 0.1, rng)                 # slowly varying control
    y = 2.0 * x + ar1(n, 0.99, 0.07, rng)       # response + correlated noise
    return np.c_[x, t], y

X, y = make_run(seed=2026)
X_new, y_new = make_run(seed=7)                 # an independent new run
model = RandomForestRegressor(200, min_samples_leaf=5, random_state=0, n_jobs=-1)
for cols in ([0], [0, 1]):                      # x only / x and t
    for name, cv in [('shuffled', KFold(5, shuffle=True, random_state=0)),
                     ('blocked',  KFold(5, shuffle=False))]:
        rmse = -cross_val_score(model, X[:, cols], y, cv=cv,
                                scoring='neg_root_mean_squared_error')
        print(cols, name, rmse.mean().round(3))
    model.fit(X[:, cols], y)
    rmse_new = np.sqrt(np.mean((model.predict(X_new[:, cols]) - y_new)**2))
    print(cols, 'new run', rmse_new.round(3))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    noise_sd = 0.07 / np.sqrt(1 - 0.99**2)
    print(f"\nstationary sd of the correlated noise: {noise_sd:.3f} (floor for any model on a new run); "
          f"sample sd in this run {(y - 2 * X[:, 0]).std():.3f}")

## `02_information_criteria.py` — โค้ดที่ 5.6

Ch.9  AIC, BIC and blocked 10-fold CV for four oscillator models on the Chapter 5 data (tab:ic-damped);

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_information_criteria.py  จากโฟลเดอร์ของบท
__file__ = '02_information_criteria.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  AIC, BIC and blocked 10-fold CV for four oscillator models on the Chapter 5 data (tab:ic-damped);
likelihood-ratio test M2 -> M3; the many local minima of M4.

Book references: ssec:ic, tab:ic-damped, ssec:sig-testing (Wilks), review question 3
Data: code:damped-cf of Chapter 5 (200 points, sigma 0.05 m, seed 2026), executed from ../ch05_regression.
"""
import os
import warnings
from pathlib import Path

import numpy as np
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.model_selection import KFold

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
warnings.filterwarnings('ignore')
_s = (HERE.parent / 'ch05_regression' / '04_damped_oscillator.py').read_text()
exec(_s[_s.index('\n# --- code:damped-cf'):_s.index('\n# --- numbers quoted')])    # t, x_obs, sigma_x, ...


def M1(t, A, w, p): return A * np.cos(w * t + p)                                     # undamped
def M2(t, A, g, w, p): return A * np.exp(-g * t) * np.cos(w * t + p)                 # damped (true form)
def M3(t, A, g, w, p, c): return M2(t, A, g, w, p) + c                               # + zero offset
def M4(t, A, g, w, p, B, g2, w2, p2): return M2(t, A, g, w, p) + M2(t, B, g2, w2, p2)  # two damped terms


MODELS = {'M1': (M1, [1, 6, 0], ([0, 0, -np.pi], [10, 20, np.pi])),
          'M2': (M2, [1, .2, 6, 0], ([0, 0, 0, -np.pi], [10, 5, 20, np.pi])),
          'M3': (M3, [1, .2, 6, 0, 0], ([0, 0, 0, -np.pi, -1], [10, 5, 20, np.pi, 1])),
          'M4': (M4, None, ([0, 0, 0, -np.pi, 0, -5, 0, -np.pi], [10, 5, 20, np.pi, 10, 5, 20, np.pi]))}


def fit(f, p0, bounds, tt, xx, sig):
    return curve_fit(f, tt, xx, p0=p0, bounds=bounds, sigma=sig * np.ones(tt.size), absolute_sigma=True, maxfev=20000)


def fit_m4_multistart(tt, xx, sig):
    """M4 has many local minima: start from M2's best fit plus a small second term at ~400 starting points."""
    base = fit(M2, [1, .2, 6, 0], MODELS['M2'][2], tt, xx, sig)[0]
    found = []
    for w2 in np.arange(0.2, 20, 0.2):
        for g2 in (-0.3, -0.05, 0.05, 0.5):
            for p2 in (-2, 0, 2):
                try:
                    p, c = fit(M4, list(base) + [0.03, g2, w2, p2], MODELS['M4'][2], tt, xx, sig)
                except (RuntimeError, ValueError):
                    continue
                found.append((np.sum(((xx - M4(tt, *p)) / sig)**2), p, c))
    found.sort(key=lambda r: r[0])
    return found


def table(tt, xx, sig, label):
    n = tt.size
    rows, extra = {}, {}
    m4 = fit_m4_multistart(tt, xx, sig)
    for name, (f, p0, bounds) in MODELS.items():
        if name == 'M4':
            chi2, p, c = m4[0]
        else:
            p, c = fit(f, p0, bounds, tt, xx, sig)
            chi2 = np.sum(((xx - f(tt, *p)) / sig)**2)
        k = len(p)
        cv = {}
        for cv_name, splitter in [('blocked', KFold(10)), ('shuffled', KFold(10, shuffle=True, random_state=0))]:
            err = []
            for tr, te in splitter.split(tt):
                try:
                    pp, _ = curve_fit(f, tt[tr], xx[tr], p0=p, bounds=bounds, maxfev=20000)
                except RuntimeError:
                    pp = p
                err.append(np.mean((xx[te] - f(tt[te], *pp))**2))
            cv[cv_name] = np.sqrt(np.mean(err))
        rows[name] = (k, chi2, chi2 / (n - k), chi2 + 2 * k, chi2 + k * np.log(n), cv['blocked'], cv['shuffled'])
        extra[name] = (p, np.sqrt(np.diag(c)))
    print(f"\n{label}")
    print("model  k   chi2_min  chi2_red     AIC      BIC   CV blocked  CV shuffled")
    for name, r in rows.items():
        print(f"{name:5s} {r[0]:2d} {r[1]:9.1f} {r[2]:8.2f} {r[3]:8.1f} {r[4]:8.1f} {r[5]:10.4f} {r[6]:11.4f}")
    minima = np.unique(np.round([r[0] for r in m4], 1))
    print(f"M4 local minima found: {len(minima)} distinct chi2 values from {minima.min():.1f} to {minima[minima < minima.min() + 10].max():.1f} (+ worse ones)")
    p4 = extra['M4'][0]
    print(f"best M4 second term: B = {p4[4]:.3f} m, gamma2 = {p4[5]:.3f} 1/s, omega2 = {p4[6]:.2f} rad/s")
    print(f"M3 offset c = {extra['M3'][0][4]:.3f} +/- {extra['M3'][1][4]:.3f} m")
    return rows


if __name__ == '__main__':
    rows = table(t, x_obs, sigma_x, "Chapter 5 data: 200 points in 10 s, sigma 0.05 m (tab:ic-damped)")
    d23 = rows['M2'][1] - rows['M3'][1]
    print(f"\nWilks test M2 -> M3: delta chi2 = {d23:.1f}, 1 dof, p = {stats.chi2.sf(d23, 1):.2f}")
    print(f"AIC(M2) - AIC(M4) = {rows['M2'][3] - rows['M4'][3]:.1f}; BIC(M4) - BIC(M2) = {rows['M4'][4] - rows['M2'][4]:.1f}; "
          f"BIC(M3) - BIC(M2) = {rows['M3'][4] - rows['M2'][4]:.1f}")
    print(f"chi2 drop M2 -> M4: {rows['M2'][1] - rows['M4'][1]:.1f}; AIC penalty for 4 extra parameters 8, BIC penalty {4 * np.log(200):.1f}")

## `03_nested_cv.py` — โค้ดที่ 9.2

Ch.9  Optimistic bias of the best grid-search score vs nested CV, on pure-noise labels (20 data sets).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_nested_cv.py  จากโฟลเดอร์ของบท
__file__ = '03_nested_cv.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Optimistic bias of the best grid-search score vs nested CV, on pure-noise labels (20 data sets).

Book references: ssec:cv-select, code:nested-cv (line for line).  About 1 minute.
numpy and cross_val_score come from code:blocked-cv (01_blocked_cv.py); they are imported here directly.
"""
import os
from pathlib import Path

import numpy as np
from sklearn.model_selection import cross_val_score

os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 9.2

In [ ]:
# --- code:nested-cv ----------------------------------------------------------
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

res = []
for seed in range(20):                           # 20 independent data sets
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(60, 20))
    y = rng.integers(0, 2, 60)                   # labels are pure noise
    grid = {'svc__C': np.logspace(-2, 3, 6), 'svc__gamma': np.logspace(-4, 1, 6)}
    inner = StratifiedKFold(5, shuffle=True, random_state=seed)
    outer = StratifiedKFold(5, shuffle=True, random_state=seed + 100)
    gs = GridSearchCV(make_pipeline(StandardScaler(), SVC()), grid,
                      cv=inner, scoring='roc_auc')
    gs.fit(X, y)
    nested = cross_val_score(gs, X, y, cv=outer, scoring='roc_auc').mean()
    res.append((gs.best_score_, nested))
print(np.mean(res, axis=0).round(3))             # [best inner score, nested]

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    r = np.array(res)
    print(f"best inner score > 0.5 in {np.sum(r[:, 0] > 0.5)} of 20 data sets; grid of {6 * 6} (C, gamma) pairs")

## `04_bootstrap_q.py` — โค้ดที่ 9.3

Ch.9  Pairs bootstrap of the damped-oscillator fit (code:bootstrap) and the Q table (tab:boot-q):

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_bootstrap_q.py  จากโฟลเดอร์ของบท
__file__ = '04_bootstrap_q.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Pairs bootstrap of the damped-oscillator fit (code:bootstrap) and the Q table (tab:boot-q):
linear propagation, pairs bootstrap and parametric bootstrap for good (200 pts, 0.05 m) and poor (60 pts, 0.4 m) data.

Book references: ssec:nonparam-boot, ssec:param-boot, code:bootstrap (line for line), tab:boot-q
The listing continues from code:damped-cf of Chapter 5, which is executed first.
"""
import os
import warnings
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
warnings.filterwarnings('ignore')
_s = (HERE.parent / 'ch05_regression' / '04_damped_oscillator.py').read_text()
exec(_s[_s.index('\n# --- code:damped-cf'):_s.index('\n# --- numbers quoted')])    # popt, pcov, t, x_obs, bounds

### โค้ดที่ 9.3

In [ ]:
# --- code:bootstrap ----------------------------------------------------------
# continues from the damped-oscillator fit (popt, pcov, t, x_obs, bounds)
rng = np.random.default_rng(seed=1)
n_boot = 1000
params_boot = []
for _ in range(n_boot):
    idx = rng.integers(0, len(t), len(t))          # resample pairs (t_i, x_i)
    try:
        p_b, _ = curve_fit(damped_oscillator, t[idx], x_obs[idx],
                           p0=popt, bounds=bounds)
        params_boot.append(p_b)
    except RuntimeError:                           # rare non-convergence
        pass
params_boot = np.array(params_boot)
Q_boot = params_boot[:, 2] / (2 * params_boot[:, 1])
print("SE(gamma), SE(omega):", params_boot[:, 1:3].std(axis=0).round(4))
print("Q 95% percentile interval:", np.percentile(Q_boot, [2.5, 97.5]).round(2))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------


def q_intervals(n, sig, n_boot=1000):
    rng_d = np.random.default_rng(seed=2026)
    tt = np.linspace(0, 10, n)
    xx = damped_oscillator(tt, 1.0, 0.3, 6.0, 0.5) + rng_d.normal(scale=sig, size=n)
    p, c = curve_fit(damped_oscillator, tt, xx, p0=[1.0, 0.2, 6.0, 0.0], bounds=bounds,
                     sigma=sig * np.ones(n), absolute_sigma=True)
    Q = p[2] / (2 * p[1])
    J = np.array([0, -p[2] / (2 * p[1]**2), 1 / (2 * p[1]), 0])
    sQ = np.sqrt(J @ c @ J)
    r = np.random.default_rng(1); pairs = []
    for _ in range(n_boot):
        i = r.integers(0, n, n)
        try:
            pairs.append(curve_fit(damped_oscillator, tt[i], xx[i], p0=p, bounds=bounds)[0])
        except RuntimeError:
            pass
    r = np.random.default_rng(1); param = []
    for _ in range(n_boot):
        xs = damped_oscillator(tt, *p) + r.normal(scale=sig, size=n)
        try:
            param.append(curve_fit(damped_oscillator, tt, xs, p0=p, bounds=bounds,
                                   sigma=sig * np.ones(n), absolute_sigma=True)[0])
        except RuntimeError:
            pass
    pairs, param = np.array(pairs), np.array(param)
    return dict(Q=Q, gamma=(p[1], np.sqrt(c[1, 1])), linear=(Q - 1.96 * sQ, Q + 1.96 * sQ),
                pairs=np.percentile(pairs[:, 2] / (2 * pairs[:, 1]), [2.5, 97.5]),
                param=np.percentile(param[:, 2] / (2 * param[:, 1]), [2.5, 97.5]), n_ok=(len(pairs), len(param)))


if __name__ == '__main__':
    print("curve_fit SE(gamma), SE(omega):", np.sqrt(np.diag(pcov))[1:3].round(4))
    print("\ntab:boot-q (true Q = 10):")
    for n, sig in [(200, 0.05), (60, 0.4)]:
        r = q_intervals(n, sig)
        print(f"  {n:3d} points, sigma {sig}: best Q {r['Q']:.2f}, gamma {r['gamma'][0]:.3f} +/- {r['gamma'][1]:.3f}; "
              f"linear [{r['linear'][0]:.2f}, {r['linear'][1]:.2f}]; pairs [{r['pairs'][0]:.2f}, {r['pairs'][1]:.2f}]; "
              f"parametric [{r['param'][0]:.2f}, {r['param'][1]:.2f}]  (fits converged {r['n_ok']})")

## `05_mc_tau.py` — โค้ดที่ 9.4

Ch.9  Integrated autocorrelation time, block bootstrap and blocking for Metropolis energies of the 2D Ising model.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_mc_tau.py  จากโฟลเดอร์ของบท
__file__ = '05_mc_tau.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Integrated autocorrelation time, block bootstrap and blocking for Metropolis energies of the 2D Ising model.

Book references: ssec:boot-mc-primary, def:tau-int, code:mc-tau (line for line), tab:mc-tau
init_lattice, total_energy and metropolis_step come from code:metropolis of Chapter 4 (executed from
../ch04_statistics, without its temperature scan).  Each temperature takes about 45 s in pure Python;
the script runs T = 2.3 (the listing) and then T = 2.0 and 3.0 for the table: about 2.5 minutes.
"""
import os
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_s = (HERE.parent / 'ch04_statistics' / '07_metropolis_ising.py').read_text()
_s = _s[_s.index('\n# --- code:metropolis'):]
exec(_s[:_s.index('def simulate_T')])                                   # rng, init_lattice, total_energy, metropolis_step

### โค้ดที่ 9.4

In [ ]:
# --- code:mc-tau -------------------------------------------------------------
# uses init_lattice, total_energy and metropolis_step from the Chapter 4 listing
def energy_series(L, T, n_thermalize, n_measure):
    lattice = init_lattice(L)
    beta = 1.0 / T
    for _ in range(n_thermalize):
        metropolis_step(lattice, beta)
    E = np.empty(n_measure)
    for k in range(n_measure):
        metropolis_step(lattice, beta)
        E[k] = total_energy(lattice)
    return E

def tau_int(x, c=5.0):
    """Integrated autocorrelation time with Sokal's automatic window."""
    x = np.asarray(x) - np.mean(x)
    n = len(x)
    f = np.fft.rfft(x, 2 * n)                      # zero-padded FFT
    acf = np.fft.irfft(f * np.conj(f))[:n]
    acf /= acf[0]
    tau = 0.5
    for W in range(1, n):
        tau += acf[W]
        if W >= c * tau:
            break
    return tau, W

def block_bootstrap(x, stat, block, n_boot=2000, rng=None):
    rng = rng or np.random.default_rng(0)
    n_blocks = len(x) // block
    blocks = np.asarray(x)[:n_blocks * block].reshape(n_blocks, block)
    out = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, n_blocks, n_blocks)   # resample whole blocks
        out[b] = stat(blocks[pick].ravel())
    return out

rng = np.random.default_rng(seed=2026)             # restart the generator
L, T = 16, 2.3
e = energy_series(L, T, n_thermalize=2000, n_measure=20000) / L**2
tau, W = tau_int(e)
se_naive = e.std(ddof=1) / np.sqrt(len(e))
print(f"<e> = {e.mean():.4f}, tau_int = {tau:.1f} sweeps (window {W})")
print(f"SE naive = {se_naive:.5f}, SE with tau = {se_naive*np.sqrt(2*tau):.5f}")
cv = lambda x: x.var() * L**2 / T**2               # C_v per spin
for stat, name in [(np.mean, '<e>'), (cv, 'C_v/N')]:
    boot = block_bootstrap(e, stat, block=200)
    print(name, "block-bootstrap SE", boot.std().round(5),
          "95% CI", np.percentile(boot, [2.5, 97.5]).round(4))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------


def blocking(x, min_blocks=16):
    """Flyvbjerg-Petersen: SE of the mean after repeatedly averaging neighbouring pairs."""
    x = np.asarray(x, float); out = []
    while len(x) >= min_blocks:
        out.append(np.sqrt(x.var(ddof=1) / len(x)))
        m = len(x) // 2 * 2
        x = 0.5 * (x[:m:2] + x[1:m:2])
    return np.array(out)


def row(e_series, T):
    tau_T, _ = tau_int(e_series)
    naive = e_series.std(ddof=1) / np.sqrt(len(e_series))
    cvT = lambda x: x.var() * L**2 / T**2
    return (e_series.mean(), tau_T, naive, naive * np.sqrt(2 * tau_T),
            block_bootstrap(e_series, np.mean, block=200).std(),
            block_bootstrap(e_series, cvT, block=1).std(), block_bootstrap(e_series, cvT, block=200).std(), cvT(e_series))


if __name__ == '__main__':
    print(f"N_eff = {len(e) / (2 * tau):.0f} of {len(e)}; corrected / naive SE = {np.sqrt(2 * tau):.1f}")
    print("blocking SE of <e> (levels 0, 1, 2, ...):", blocking(e).round(5))
    for blk in (50, 100, 200, 500, 1000):
        print(f"block bootstrap SE of <e>, blocks of {blk:4d}: {block_bootstrap(e, np.mean, block=blk).std():.5f}")
    print(f"C_v/N = {cv(e):.3f}; single-point bootstrap SE {block_bootstrap(e, cv, block=1).std():.4f}")

    rows = {2.3: row(e, 2.3)}
    for T_other in (2.0, 3.0):
        rng = np.random.default_rng(seed=2026)                          # same restart as in the listing
        e_T = energy_series(L, T_other, n_thermalize=2000, n_measure=20000) / L**2
        rows[T_other] = row(e_T, T_other)
    print("\ntab:mc-tau   T   <e>      tau_int  SE naive  SE tau   SE block | C_v/N  SE single  SE block")
    for T_val in (2.0, 2.3, 3.0):
        r = rows[T_val]
        print(f"           {T_val:.1f} {r[0]:8.4f} {r[1]:6.1f} {r[2]:9.5f} {r[3]:8.4f} {r[4]:8.4f} | {r[7]:5.3f} {r[5]:9.4f} {r[6]:9.4f}")

## `06_wang_landau_seeds.py` — โค้ดที่ 4.7

Ch.9  Run-to-run spread of Wang-Landau for the 4x4 Ising model: ten independent runs (seeds 0-9, ln f_final = 1e-6)

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_wang_landau_seeds.py  จากโฟลเดอร์ของบท
__file__ = '06_wang_landau_seeds.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Run-to-run spread of Wang-Landau for the 4x4 Ising model: ten independent runs (seeds 0-9, ln f_final = 1e-6)
compared with the exact C_v/N peak from enumerating all 2^16 states.

Book references: ssec:wl-secondary; code:wl of Chapter 4 (executed from ../ch04_statistics).  About 4 minutes on one core
(runs in parallel when several cores are available).
"""
import itertools
import os
from multiprocessing import Pool
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_s = (HERE.parent / 'ch04_statistics' / '08_wang_landau_ising.py').read_text()
exec(_s[_s.index('# from code:metropolis'):_s.index('E_bins, log_g, n_sweeps = wang_landau_2d_ising')])
T_GRID = np.arange(1.5, 3.5, 0.001)


def cv_peak(E, lg):
    cvs = np.array(thermodynamic(E, lg, T_GRID)['Cv']) / 16
    k = cvs.argmax()
    return cvs[k], T_GRID[k]


def one_run(seed):
    global rng
    rng = np.random.default_rng(seed)                 # the listing draws from the global rng
    E, lg, sweeps = wang_landau_2d_ising(L=4, ln_f_final=1e-6)
    return seed, cv_peak(E, lg), sweeps


if __name__ == '__main__':
    counts = {}
    for bits in itertools.product([-1, 1], repeat=16):
        E = total_energy(np.array(bits).reshape(4, 4))
        counts[E] = counts.get(E, 0) + 1
    Ex = np.array(sorted(counts))
    peak_x, T_x = cv_peak(Ex, np.log([counts[k] for k in Ex]))
    print(f"exact: C_v/N peak {peak_x:.4f} at T = {T_x:.3f}")
    with Pool(min(10, os.cpu_count() or 1)) as pool:
        res = pool.map(one_run, range(10))
    pk = np.array([r[1][0] for r in res]); Tp = np.array([r[1][1] for r in res])
    for s, (p, Tpk), sw in res:
        print(f"  seed {s}: peak {p:.4f} at T = {Tpk:.3f} ({sw} sweeps)")
    print(f"single runs {pk.min():.3f}-{pk.max():.3f}; mean {pk.mean():.4f} +/- {pk.std(ddof=1) / np.sqrt(10):.4f} "
          f"(sd {pk.std(ddof=1):.4f}); peak T {Tp.mean():.3f} +/- {Tp.std(ddof=1) / np.sqrt(10):.3f}")

## `07_mcmc_hand.py` — โค้ดที่ 9.5

Ch.9  Hand-written Metropolis sampler for the damped-oscillator posterior (uniform prior inside the bounds, known sigma).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_mcmc_hand.py  จากโฟลเดอร์ของบท
__file__ = '07_mcmc_hand.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Hand-written Metropolis sampler for the damped-oscillator posterior (uniform prior inside the bounds, known sigma).

Book references: ssec:mcmc, code:mcmc-hand (line for line)
Continues from code:damped-cf (Chapter 5) and uses tau_int from code:mc-tau (05_mc_tau.py, function definitions only).
"""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_s = (HERE.parent / 'ch05_regression' / '04_damped_oscillator.py').read_text()
exec(_s[_s.index('\n# --- code:damped-cf'):_s.index('\n# --- numbers quoted')])      # popt, pcov, t, x_obs, sigma_x, bounds
_s = (HERE / '05_mc_tau.py').read_text()
exec(_s[_s.index('def tau_int'):_s.index('def block_bootstrap')])                      # tau_int only

### โค้ดที่ 9.5

In [ ]:
# --- code:mcmc-hand ----------------------------------------------------------
# continues from the damped-oscillator fit (popt, pcov, t, x_obs, sigma_x, bounds)
lo, hi = np.array(bounds[0]), np.array(bounds[1])

def log_posterior(theta):
    if np.any(theta < lo) or np.any(theta > hi):      # uniform prior inside bounds
        return -np.inf
    chi2 = np.sum(((x_obs - damped_oscillator(t, *theta)) / sigma_x)**2)
    return -0.5 * chi2                                 # ln L + const

rng = np.random.default_rng(seed=1)
step = 2.38**2 / 4 * pcov                              # proposal covariance
theta, lp = popt.copy(), log_posterior(popt)
chain, n_acc = [], 0
for k in range(50000):
    prop = rng.multivariate_normal(theta, step)
    lp_prop = log_posterior(prop)
    if np.log(rng.random()) < lp_prop - lp:            # Metropolis rule, T = 1
        theta, lp = prop, lp_prop
        n_acc += 1
    chain.append(theta)
chain = np.array(chain)[5000:]                         # drop burn-in
print("acceptance:", n_acc / 50000)
for name, c in zip(['A', 'gamma', 'omega', 'phi'], chain.T):
    lo95, med, hi95 = np.percentile(c, [2.5, 50, 97.5])
    print(f"{name:5s} median {med:.4f}  95% [{lo95:.4f}, {hi95:.4f}]  "
          f"tau_int {tau_int(c)[0]:.1f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(f"posterior sd: gamma {chain[:, 1].std():.4f}, omega {chain[:, 2].std():.4f}; curve_fit SE "
          f"{np.sqrt(pcov[1, 1]):.4f}, {np.sqrt(pcov[2, 2]):.4f}")
    taus = [tau_int(c)[0] for c in chain.T]
    print(f"independent samples ~ {len(chain) / (2 * np.mean(taus)):.0f} of {len(chain)} (1 in {2 * np.mean(taus):.0f})")

## `08_pymc_oscillator.py` — โค้ดที่ 9.6

Ch.9  Bayesian fit of the damped oscillator with PyMC (NUTS), posterior predictive checks,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 08_pymc_oscillator.py  จากโฟลเดอร์ของบท
__file__ = '08_pymc_oscillator.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Bayesian fit of the damped oscillator with PyMC (NUTS), posterior predictive checks,
and PSIS-LOO / WAIC comparison of M1, M2, M3.

Book references: ssec:mcmc, code:pymc (line for line), ssec:ppc, ssec:waic-loo.  Needs pymc and arviz.
Continues from code:damped-cf (Chapter 5).  The first run compiles PyTensor code (can take minutes);
after that about 1 minute (four models).
"""
import logging
import os
import warnings
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
warnings.filterwarnings('ignore')
_s = (HERE.parent / 'ch05_regression' / '04_damped_oscillator.py').read_text()
exec(_s[_s.index('\n# --- code:damped-cf'):_s.index('\n# --- numbers quoted')])      # t, x_obs

### โค้ดที่ 9.6

In [ ]:
# --- code:pymc ---------------------------------------------------------------
import pymc as pm
import arviz as az

with pm.Model() as model:
    A     = pm.HalfNormal('A', sigma=2.0)                  # A > 0
    gamma = pm.HalfNormal('gamma', sigma=1.0)              # gamma >= 0
    omega = pm.TruncatedNormal('omega', mu=6.0, sigma=1.0, lower=0.0)  # from k and m
    phi   = pm.Uniform('phi', lower=-np.pi, upper=np.pi)
    sigma = pm.HalfNormal('sigma', sigma=0.2)              # noise level, unknown
    mu = A * pm.math.exp(-gamma * t) * pm.math.cos(omega * t + phi)
    pm.Normal('x', mu=mu, sigma=sigma, observed=x_obs)
    idata = pm.sample(2000, tune=1000, chains=4, random_seed=2026,
                      idata_kwargs={'log_likelihood': True})
    pm.sample_posterior_predictive(idata, extend_inferencedata=True,
                                   random_seed=2026)
print(az.summary(idata, var_names=['A', 'gamma', 'omega', 'phi', 'sigma'],
                 hdi_prob=0.95))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    logging.getLogger('pymc').setLevel(logging.ERROR)
    post = idata.posterior.stack(s=('chain', 'draw'))
    A_, g_, w_, p_, s_ = (post[v].values for v in ('A', 'gamma', 'omega', 'phi', 'sigma'))
    print(f"\ngamma posterior sd {g_.std():.4f}; ess_bulk "
          f"{az.summary(idata, var_names=['A', 'gamma', 'omega', 'phi', 'sigma'])['ess_bulk'].agg(['min', 'max']).tolist()}")
    mu = A_[:, None] * np.exp(-g_[:, None] * t) * np.cos(w_[:, None] * t + p_[:, None])
    yrep = idata.posterior_predictive['x'].stack(s=('chain', 'draw')).values.T

    def ppc_stats(y):
        r = (y - mu) / s_[:, None]
        return (r**2).sum(1), np.abs(r).max(1), np.array([np.corrcoef(ri[:-1], ri[1:])[0, 1] for ri in r])

    obs, rep = ppc_stats(np.broadcast_to(x_obs, mu.shape)), ppc_stats(yrep)
    for name, o, r in zip(['sum (resid/sigma)^2', 'max |resid/sigma|', 'lag-1 resid correlation'], obs, rep):
        print(f"posterior predictive p-value, {name:24s}: {(r >= o).mean():.2f}")

    def fit_model(kind):
        with pm.Model():
            A = pm.HalfNormal('A', sigma=2.0)
            omega = pm.TruncatedNormal('omega', mu=6.0, sigma=1.0, lower=0.0)
            phi = pm.Uniform('phi', lower=-np.pi, upper=np.pi)
            sigma = pm.HalfNormal('sigma', sigma=0.2)
            if kind == 'M1':
                mu_ = A * pm.math.cos(omega * t + phi)
            else:
                gamma = pm.HalfNormal('gamma', sigma=1.0)
                mu_ = A * pm.math.exp(-gamma * t) * pm.math.cos(omega * t + phi)
                if kind == 'M3':
                    mu_ = mu_ + pm.Normal('c', mu=0.0, sigma=0.5)       # zero offset, weak prior
            pm.Normal('x', mu=mu_, sigma=sigma, observed=x_obs)
            return pm.sample(2000, tune=1000, chains=4, random_seed=2026, progressbar=False,
                             idata_kwargs={'log_likelihood': True})

    ids = {'M1': fit_model('M1'), 'M2': idata, 'M3': fit_model('M3')}
    for ic in ('loo', 'waic'):
        print(f"\naz.compare, {ic}:")
        print(az.compare(ids, ic=ic).iloc[:, :7].round(2).to_string())
    print("max Pareto k:", {k: round(float(az.loo(v, pointwise=True).pareto_k.max()), 2) for k, v in ids.items()})

## `09_delong_calibration.py` — โค้ดที่ 9.7, 9.8

Ch.9  Paired comparison of classifiers (DeLong test, paired bootstrap, McNemar) and calibration of the

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 09_delong_calibration.py  จากโฟลเดอร์ของบท
__file__ = '09_delong_calibration.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Paired comparison of classifiers (DeLong test, paired bootstrap, McNemar) and calibration of the
balanced random forest (Brier, ECE, isotonic), plus the prior-shift correction to a 1 % signal fraction.

Book references: ssec:sig-testing, ssec:calibration, code:delong, code:calibration (line for line)
Continues from the Chapter 6 listings (toy Higgs data, classifiers, rf_pipe), executed from ../ch06_classification.
"""
import os
import warnings
from pathlib import Path

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
warnings.filterwarnings('ignore')
_CH6 = HERE.parent / 'ch06_classification'
for _f in ('01_higgs_toy_data.py', '02_logreg_forest.py', '03_compare_five.py'):
    __file__ = str(_CH6 / _f)                                  # Chapter 6 listings chdir via __file__
    _s = (_CH6 / _f).read_text()
    exec(_s[_s.index('\n# --- code:'):_s.index('\n# --- end of listings')])
__file__ = str(HERE / '09_delong_calibration.py')
os.chdir(HERE)

### โค้ดที่ 9.7

In [ ]:
# --- code:delong -------------------------------------------------------------
from scipy import stats

def delong(y, s1, s2):
    """Paired test for AUC(s1) - AUC(s2) on the same events."""
    pos, neg = y == 1, y == 0
    def components(s):
        sp, sn = s[pos], s[neg]
        psi = (sp[:, None] > sn[None, :]) + 0.5 * (sp[:, None] == sn[None, :])
        return psi.mean(), psi.mean(axis=1), psi.mean(axis=0)
    a1, v10_1, v01_1 = components(s1)
    a2, v10_2, v01_2 = components(s2)
    S10 = np.cov(np.vstack([v10_1, v10_2]))           # over signal events
    S01 = np.cov(np.vstack([v01_1, v01_2]))           # over background events
    S = S10 / pos.sum() + S01 / neg.sum()
    diff = a1 - a2
    se = np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])     # covariance term matters
    return diff, se, 2 * stats.norm.sf(abs(diff) / se)

score = {}                                        # classifiers: Chapter 6 listing
for name in ['KNN', 'SVM (RBF)', 'RandForest']:
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', classifiers[name])])
    pipe.fit(X_train, y_train)
    score[name] = (pipe.decision_function(X_val) if name == 'SVM (RBF)'
                   else pipe.predict_proba(X_val)[:, 1])
d, se, p = delong(y_val, score['SVM (RBF)'], score['RandForest'])
print(f"AUC(SVM) - AUC(RF) = {d:.4f} +/- {se:.4f}, p = {p:.3f}")

### โค้ดที่ 9.8

In [ ]:
# --- code:calibration --------------------------------------------------------
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

def ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean())
               for b in range(bins) if np.any(idx == b))

rf_cal = CalibratedClassifierCV(rf_pipe, method='isotonic', cv=5)
for name, model in [('balanced RF', rf_pipe), ('+ isotonic', rf_cal)]:
    model.fit(X_train, y_train)
    p = model.predict_proba(X_val)[:, 1]
    frac_pos, mean_p = calibration_curve(y_val, p, n_bins=10)
    print(name, "Brier", round(brier_score_loss(y_val, p), 4),
          "ECE", round(ece(y_val, p), 4))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    from sklearn.metrics import roc_auc_score
    pos, neg = y_val == 1, y_val == 0

    def auc_se(s):
        psi = (s[pos][:, None] > s[neg][None, :]) + 0.5 * (s[pos][:, None] == s[neg][None, :])
        return np.sqrt(psi.mean(1).var(ddof=1) / pos.sum() + psi.mean(0).var(ddof=1) / neg.sum())

    print()
    for n, s in score.items():
        print(f"{n:11s} AUC {roc_auc_score(y_val, s):.4f}  SE {auc_se(s):.4f}")
    print(f"unpaired SE of the difference SVM - RF: {np.hypot(auc_se(score['SVM (RBF)']), auc_se(score['RandForest'])):.4f}")
    d2, se2, p2 = delong(y_val, score['RandForest'], score['KNN'])
    print(f"AUC(RF) - AUC(KNN) = {d2:.4f} +/- {se2:.4f}, p = {p2:.2f}")
    r = np.random.default_rng(0); D = []
    for _ in range(1000):
        i = r.integers(0, len(y_val), len(y_val))
        D.append(roc_auc_score(y_val[i], score['SVM (RBF)'][i]) - roc_auc_score(y_val[i], score['RandForest'][i]))
    print("paired bootstrap 95 % CI of AUC(SVM) - AUC(RF):", np.percentile(D, [2.5, 97.5]).round(4))

    pred = {}
    for n in ('KNN', 'RandForest'):
        pred[n] = Pipeline([('scaler', StandardScaler()), ('clf', classifiers[n])]).fit(X_train, y_train).predict(X_val)
    ok_rf, ok_knn = pred['RandForest'] == y_val, pred['KNN'] == y_val
    n01, n10 = int((ok_rf & ~ok_knn).sum()), int((~ok_rf & ok_knn).sum())
    chi = (abs(n01 - n10) - 1)**2 / (n01 + n10)
    print(f"McNemar RF vs KNN: n01 = {n01}, n10 = {n10}, p = {stats.chi2.sf(chi, 1):.3f}; accuracy RF {ok_rf.mean():.3f}, KNN {ok_knn.mean():.3f}")

    p_raw = rf_pipe.fit(X_train, y_train).predict_proba(X_val)[:, 1]
    for lo, hi in [(0.4, 0.5), (0.5, 0.6)]:
        m = (p_raw >= lo) & (p_raw < hi)
        print(f"balanced RF, scores {lo}-{hi}: mean score {p_raw[m].mean():.2f}, signal fraction {y_val[m].mean():.2f}")
    p_cal = rf_cal.predict_proba(X_val)[:, 1]
    print(f"AUC after isotonic calibration {roc_auc_score(y_val, p_cal):.4f}")

    def prior_shift(p, pi_new, pi_train=0.2):
        a = pi_new / pi_train * p
        return a / (a + (1 - pi_new) / (1 - pi_train) * (1 - p))

    print(f"events with calibrated p > 0.5: {100 * (p_cal > 0.5).mean():.1f} %; after shifting to a 1 % signal fraction: "
          f"{100 * (prior_shift(p_cal, 0.01) > 0.5).mean():.1f} %")

## `10_ensemble_spread.py` — โค้ดที่ 9.9

Ch.9  Ensemble spread inside and outside the training range: bootstrap polynomials vs random forest.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 10_ensemble_spread.py  จากโฟลเดอร์ของบท
__file__ = '10_ensemble_spread.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Ensemble spread inside and outside the training range: bootstrap polynomials vs random forest.

Book references: ssec:ensemble, code:ensemble-spread (line for line)
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

os.chdir(Path(__file__).resolve().parent)
Path('outputs').mkdir(exist_ok=True)

### โค้ดที่ 9.9

In [ ]:
# --- code:ensemble-spread ----------------------------------------------------
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(seed=2026)
x = rng.uniform(0, 3, 150)                       # training data only in [0, 3]
y = np.sin(2 * x) + rng.normal(scale=0.1, size=x.size)
x_grid = np.array([0.5, 1.5, 2.5, 3.5, 4.0])

poly = np.array([np.polyval(np.polyfit(x[i], y[i], 5), x_grid)
                 for i in (rng.integers(0, x.size, x.size) for _ in range(200))])
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                           random_state=0).fit(x[:, None], y)
trees = np.array([tree.predict(x_grid[:, None]) for tree in rf.estimators_])

print("truth      ", np.sin(2 * x_grid).round(2))
print("poly mean  ", poly.mean(0).round(2), " spread", poly.std(0).round(3))
print("forest mean", trees.mean(0).round(2), " spread", trees.std(0).round(3))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    err_rf = np.abs(trees.mean(0) - np.sin(2 * x_grid))
    print(f"forest error at x = 3.5, 4.0: {err_rf[3:].round(2)}; forest spread inside [0, 3]: "
          f"{trees.std(0)[:3].min():.3f}-{trees.std(0)[:3].max():.3f}")
    xs = np.linspace(0, 4.2, 300)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(xs, np.sin(2 * xs), 'k', label='truth'); ax.plot(x, y, '.', ms=3, alpha=0.4, label='training data')
    ax.plot(xs, rf.predict(xs[:, None]), label='random forest')
    ax.axvspan(3, 4.2, color='grey', alpha=0.15); ax.set_ylim(-2, 2); ax.legend()
    fig.tight_layout(); fig.savefig('outputs/ch09_ensemble_extrapolation.png', dpi=100)
    print("saved outputs/ch09_ensemble_extrapolation.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `11_bayes_cov_polymer.py` — โค้ดที่ 9.10

Ch.9  Posterior of the covariance of the Rouse trajectory (inverse-Wishart with N_eff): eigenvalue and eigenvector uncertainty.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 11_bayes_cov_polymer.py  จากโฟลเดอร์ของบท
__file__ = '11_bayes_cov_polymer.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.9  Posterior of the covariance of the Rouse trajectory (inverse-Wishart with N_eff): eigenvalue and eigenvector uncertainty.

Book references: ssec:cross-case, code:bayes-cov (line for line)
polymer_trajectory comes from code:rouse-gen (Chapter 8, executed from ../ch08_dimred);
tau_int from code:mc-tau (05_mc_tau.py, function definition only).
"""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_s = (HERE.parent / 'ch08_dimred' / '04_rouse_pca.py').read_text()
exec(_s[_s.index('\n# --- code:rouse-gen'):_s.index('\n# --- code:polymer-pca')])   # polymer_trajectory
_s = (HERE / '05_mc_tau.py').read_text()
exec(_s[_s.index('def tau_int'):_s.index('def block_bootstrap')])                     # tau_int only

### โค้ดที่ 9.10

In [ ]:
# --- code:bayes-cov ----------------------------------------------------------
from scipy.stats import invwishart
from sklearn.decomposition import PCA

X = polymer_trajectory - polymer_trajectory.mean(axis=0)   # Chapter 8 data
n, D = X.shape
S = X.T @ X / n                                   # sample covariance
pc1 = PCA(n_components=1).fit_transform(X)[:, 0]
tau_f, _ = tau_int(pc1)                           # in frames
n_eff = n / (2 * tau_f)                           # effectively independent frames
nu0, Psi0 = D + 2, 1e-3 * np.eye(D)               # weak, proper prior
post = invwishart(df=nu0 + n_eff, scale=Psi0 + n_eff * S)

N = 50
q1 = np.sqrt(2 / N) * np.cos(np.pi * (np.arange(N) + 0.5) / N)   # Rouse p = 1
P1 = np.kron(q1[:, None], np.eye(3))              # x, y, z copies, shape (150, 3)
rng = np.random.default_rng(seed=2026)
lam1, overlap = [], []
for Sig in post.rvs(size=1000, random_state=rng):
    w, V = np.linalg.eigh(Sig)
    lam1.append(w[-3:].mean())                    # degenerate p = 1 triplet
    overlap.append(np.sum((P1.T @ V[:, -3:])**2) / 3)
print("n_eff =", round(n_eff))
print("lambda_1 95% interval:", np.percentile(lam1, [2.5, 50, 97.5]).round(1))
print("overlap with p=1 mode:", np.percentile(overlap, [2.5, 50, 97.5]).round(3))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    print(f"tau of PC1 = {tau_f:.1f} frames (theory 84.5 / 5 = {84.46 / 5:.1f})")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E9_2_short_data_ic.py`

E9.2  Model selection with short data: the Chapter 5 oscillator (same parameters and seed) measured at 60 points in 3 s.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E9_2_short_data_ic.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E9_2_short_data_ic.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E9.2  Model selection with short data: the Chapter 5 oscillator (same parameters and seed) measured at 60 points in 3 s.
Uses the functions of 02_information_criteria.py.  About 1 minute (M4 multistart)."""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
__file__ = str(HERE / '02_information_criteria.py')
_s = (HERE / '02_information_criteria.py').read_text()
exec(_s[:_s.index("if __name__ == '__main__':")].split('"""', 2)[2])

rng = np.random.default_rng(seed=2026)
t_s = np.linspace(0, 3, 60)
x_s = damped_oscillator(t_s, 1.0, 0.3, 6.0, 0.5) + rng.normal(scale=0.05, size=t_s.size)
rows = table(t_s, x_s, 0.05, "60 points in the first 3 s, sigma 0.05 m")

## `exercises/E9_3_magnetization_tau.py` — โค้ดที่ 9.4

E9.3  tau_int of e and |m| at T = 2.3 (same seed as code:mc-tau), and the SE of <|m|> by four methods.  About 45 s.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E9_3_magnetization_tau.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E9_3_magnetization_tau.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E9.3  tau_int of e and |m| at T = 2.3 (same seed as code:mc-tau), and the SE of <|m|> by four methods.  About 45 s."""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
_s = (HERE / '05_mc_tau.py').read_text()
_ch4 = (HERE.parent / 'ch04_statistics' / '07_metropolis_ising.py').read_text()
_ch4 = _ch4[_ch4.index('\n# --- code:metropolis'):]
exec(_ch4[:_ch4.index('def simulate_T')])
_l = _s[_s.index('\n# --- code:mc-tau'):_s.index('\n# --- end of listings')]
exec(_l[:_l.index('rng = np.random.default_rng(seed=2026)')])          # energy_series, tau_int, block_bootstrap
exec(_s[_s.index('def blocking'):_s.index('def row')])


def energy_magnetisation_series(L, T, n_thermalize, n_measure):      # energy_series, also storing M
    lattice = init_lattice(L)
    beta = 1.0 / T
    for _ in range(n_thermalize):
        metropolis_step(lattice, beta)
    E, M = np.empty(n_measure), np.empty(n_measure)
    for k in range(n_measure):
        metropolis_step(lattice, beta)
        E[k] = total_energy(lattice)
        M[k] = lattice.sum()
    return E, M


rng = np.random.default_rng(seed=2026)
L, T = 16, 2.3
E, M = energy_magnetisation_series(L, T, 2000, 20000)
e, m = E / L**2, np.abs(M) / L**2
print(f"<e> = {e.mean():.4f}, tau_int(e) = {tau_int(e)[0]:.1f}")
tau_m, W = tau_int(m)
naive = m.std(ddof=1) / np.sqrt(len(m))
print(f"<|m|> = {m.mean():.3f}, tau_int(|m|) = {tau_m:.1f} (window {W})")
print(f"SE naive {naive:.4f}, with tau {naive * np.sqrt(2 * tau_m):.4f} ({np.sqrt(2 * tau_m):.1f} x)")
for blk in (100, 200, 500, 1000):
    print(f"block bootstrap, blocks of {blk:4d}: {block_bootstrap(m, np.mean, block=blk).std():.4f}")
print("blocking:", blocking(m).round(4))

## `exercises/E9_4_bad_data_mcmc.py` — โค้ดที่ 9.5

E9.4  Posterior of the poor data set (60 points in 10 s, sigma 0.4 m, seed 2026) with the sampler of code:mcmc-hand,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E9_4_bad_data_mcmc.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E9_4_bad_data_mcmc.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E9.4  Posterior of the poor data set (60 points in 10 s, sigma 0.4 m, seed 2026) with the sampler of code:mcmc-hand,
100,000 steps (burn-in 10,000).  About 10 s."""
import os
from pathlib import Path

import numpy as np
from scipy.optimize import curve_fit

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
_s = (HERE / '05_mc_tau.py').read_text()
exec(_s[_s.index('def tau_int'):_s.index('def block_bootstrap')])


def damped_oscillator(t, A, gamma, omega, phi):          # as in code:damped-cf
    return A * np.exp(-gamma * t) * np.cos(omega * t + phi)


rng = np.random.default_rng(seed=2026)
t = np.linspace(0, 10, 60)
sigma_x = 0.4
x_obs = damped_oscillator(t, 1.0, 0.3, 6.0, 0.5) + rng.normal(scale=sigma_x, size=t.size)
bounds = ([0, 0, 0, -np.pi], [10, 5, 20, np.pi])
popt, pcov = curve_fit(damped_oscillator, t, x_obs, p0=[1.0, 0.2, 6.0, 0.0], bounds=bounds,
                       sigma=sigma_x * np.ones(t.size), absolute_sigma=True)
print(f"curve_fit: gamma = {popt[1]:.3f} +/- {np.sqrt(pcov[1, 1]):.3f}, Q = {popt[2] / (2 * popt[1]):.1f}")

_l = (HERE / '07_mcmc_hand.py').read_text()
_l = _l[_l.index('\n# --- code:mcmc-hand'):_l.index('\n# --- end of listings')]
exec(_l.replace('range(50000)', 'range(100000)').replace('n_acc / 50000', 'n_acc / 100000').replace('[5000:]', '[10000:]'))
Q = chain[:, 2] / (2 * chain[:, 1])
q = np.percentile(Q, [2.5, 16, 50, 84, 97.5])
print(f"Q: median {q[2]:.1f}, 95 % [{q[0]:.1f}, {q[4]:.1f}], report {q[2]:.1f} +{q[3] - q[2]:.1f} -{q[2] - q[1]:.1f}")
print(f"independent samples of gamma: {len(chain) / (2 * tau_int(chain[:, 1])[0]):.0f}")

## `exercises/E9_5_repeated_cv.py`

E9.5  5 x 5 repeated stratified CV of KNN, SVM and random forest on the whole toy Higgs data set (same splits for all),

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E9_5_repeated_cv.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E9_5_repeated_cv.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E9.5  5 x 5 repeated stratified CV of KNN, SVM and random forest on the whole toy Higgs data set (same splits for all),
naive and Nadeau-Bengio corrected paired t-tests.  About 1-2 minutes."""
import os
import warnings
from pathlib import Path

import numpy as np
from scipy import stats
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

HERE = Path(__file__).resolve().parent.parent
warnings.filterwarnings('ignore')
_CH6 = HERE.parent / 'ch06_classification'
for _f in ('01_higgs_toy_data.py', '02_logreg_forest.py', '03_compare_five.py'):
    __file__ = str(_CH6 / _f)
    _s = (_CH6 / _f).read_text()
    exec(_s[_s.index('\n# --- code:'):_s.index('\n# --- end of listings')])
os.chdir(HERE)

X_all, y_all = df[features].to_numpy(), df['label'].to_numpy()
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=2026)
auc = {}
for name in ('SVM (RBF)', 'RandForest', 'KNN'):
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', classifiers[name])])
    auc[name] = cross_val_score(pipe, X_all, y_all, cv=cv, scoring='roc_auc', n_jobs=-1)
    print(f"{name:11s} mean AUC {auc[name].mean():.4f}")
J, ratio = 25, 1 / 4                                       # n_test / n_train for 5-fold CV
print(f"correction factor sqrt(1 + J n_test/n_train) = {np.sqrt(1 + J * ratio):.2f}")
for a, b in [('SVM (RBF)', 'RandForest'), ('SVM (RBF)', 'KNN'), ('RandForest', 'KNN')]:
    d = auc[a] - auc[b]
    t_naive = d.mean() / (d.std(ddof=1) / np.sqrt(J))
    t_nb = d.mean() / np.sqrt((1 / J + ratio) * d.var(ddof=1))
    print(f"{a} - {b}: {d.mean():+.4f} (sd {d.std(ddof=1):.4f}); naive t = {t_naive:.1f}, p = {2 * stats.t.sf(abs(t_naive), J - 1):.1g}; "
          f"corrected t = {t_nb:.1f}, p = {2 * stats.t.sf(abs(t_nb), J - 1):.1g}")